# ChuckleNet Phase A + Prosody: Frozen WavLM + Attention Pooling + Prosody Fusion
## Architecture
- **WavLM encoder** (FROZEN — this is critical! Partial unfreeze caused catastrophic overfitting)
- **Attention pooling** — learns which audio frames matter
- **21-dim prosody** (pause/F0/RMS/energy from Wav2Vec2FeatureExtractor)
- **Simple late fusion** (concatenate pooled audio + prosody → MLP)

## Key fixes from Phase D failures:
- ✅ WavLM FROZEN (no gradient → no overfitting)
- ✅ No CSA (Compressed Sparse Attention was hurting, not helping)
- ✅ No Engram bottleneck (Phase A used direct pooling → classifier)
- ✅ LR = 1e-5 for encoder (was 5e-5 in Phase D → caused shock)
- ✅ Class weight [1, 2.5] (was [1, 4] → too aggressive)
- ✅ weight_decay = 0.01
- ✅ Audio cache → RAM (was loading from Drive every batch → hang)
- ✅ padding=False in feature extractor (was True → mask mismatch)

## Expected: Val F1 ~0.75+ (Phase A frozen alone got 0.756)

In [ ]:
# ============================================================
# SETUP
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/chuckle_checkpoints'
AUDIO_DIR = f'{BASE}/audio'
AUDIO_CACHE_DIR = f'{BASE}/audio_cache'
PROSODY_PATH = f'{BASE}/prosody_phaseD.json'
OUTPUT_DIR = f'{BASE}'
AUDIO_CKPT = f'{BASE}/phaseA_prosody_best.pt'

import os, json, time, numpy as np, librosa, torch
from sklearn.metrics import f1_score, precision_score, recall_score
from collections import defaultdict
import gc

SR = 16000
MAX_LEN = 80000  # 5 seconds at 16kHz
BATCH = 32
EPOCHS = 10
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type != 'cuda':
    print('⚠️  WARNING: No GPU! Training will be very slow.')
    print('Consider: Runtime → Change runtime type → GPU')


In [ ]:
# ============================================================
# LOAD DATASET — aligned_utterances.jsonl
# ============================================================
# Format: {video_id, text, start, end, label_any, label_majority, audio_file}
# Held-out test: Bill Burr, Dave Chappelle, Russell Peters

HELD_OUT = {'BFIHCzw3itk', 'BAD4askmGgk', '1Nb3_os4RSA'}

print('Loading aligned_utterances...')
au_path = f'{BASE}/aligned_utterances.jsonl'
all_data = [json.loads(line) for line in open(au_path)]
print(f'  Total: {len(all_data)}')

# Split by video ID (held-out comedians for test)
held_out = [d for d in all_data if d['video_id'] in HELD_OUT]
in_dist = [d for d in all_data if d['video_id'] not in HELD_OUT]
print(f'  In-dist: {len(in_dist)}, Held-out test: {len(held_out)}')

# 85/15 train/val on in-distribution only
np.random.seed(SEED)
np.random.shuffle(in_dist)
n_train = int(len(in_dist) * 0.85)
train, val = in_dist[:n_train], in_dist[n_train:]
test = held_out

print(f'  Train: {len(train)}, Val: {len(val)}, Test: {len(test)} (held-out)')
print(f'  Train pos%={100*sum(d["label_any"] for d in train)/len(train):.1f}%')
print(f'  Val pos%={100*sum(d["label_any"] for d in val)/len(val):.1f}%')
print(f'  Test pos%={100*sum(d["label_any"] for d in test)/len(test):.1f}%')

# Quick sanity check
assert len(set(d['video_id'] for d in train) & set(d['video_id'] for d in held_out)) == 0, 'Leak!'
print('  ✅ No held-out leakage')


In [ ]:
# ============================================================
# LOAD PROSODY CACHE
# ============================================================
print('Loading prosody cache...')
with open(PROSODY_PATH) as f:
    prosody_raw = json.load(f)
prosody_map = {d['uid']: d['feats'] for d in prosody_raw}
print(f'  {len(prosody_map)} prosody entries, dim={len(prosody_raw[0]["feats"])}')

# Check coverage
matched = sum(1 for d in train+val+test if f"{d['video_id']}_{d['start']:.2f}" in prosody_map)
print(f'  Prosody coverage: {matched}/{len(train+val+test)} ({100*matched/(len(train+val+test)):.1f}%)')

# Normalize prosody features using StandardScaler (fit on train only!)
from sklearn.preprocessing import StandardScaler
train_prosody = np.array([prosody_map.get(f"{d['video_id']}_{d['start']:.2f}", np.zeros(21)) for d in train])
prosody_scaler = StandardScaler()
prosody_scaler.fit(train_prosody)
print(f'  Prosody scaler fitted on {len(train_prosody)} train samples')


In [ ]:
# ============================================================
# AUDIO CACHE — Load all audio into RAM
# ============================================================
print('Caching audio in RAM (~2-3 min)...')

from collections import defaultdict

_loaded_vids = {}
for vid in sorted(set(d['video_id'] for d in train + val + test)):
    audio_path = f'{AUDIO_DIR}/{vid}.mp3'
    try:
        full, _ = librosa.load(audio_path, sr=SR, mono=True)
        _loaded_vids[vid] = full
    except Exception as e:
        print(f'  Warning: could not load {vid}: {e}')
        _loaded_vids[vid] = np.zeros(int(120*SR), dtype=np.float32)

print(f'  Loaded {len(_loaded_vids)} video files')

_audio_cache = {}
for d in train + val + test:
    vid, start = d['video_id'], d['start']
    key = f'{vid}_{start:.2f}'
    try:
        s = int(max(0, start - 0.05) * SR)
        e = s + MAX_LEN
        seg = _loaded_vids[vid][s:e]
        if len(seg) < MAX_LEN:
            seg = np.pad(seg, (0, MAX_LEN - len(seg)))
        _audio_cache[key] = seg
    except:
        _audio_cache[key] = np.zeros(MAX_LEN, dtype=np.float32)

del _loaded_vids
gc.collect()
print(f'  Cached {len(_audio_cache)} segments ({len(_audio_cache)*MAX_LEN*4/1024**2:.0f} MB)')
print('Audio cache ready.\n')

def load_audio(key):
    return _audio_cache.get(key, np.zeros(MAX_LEN, dtype=np.float32))


In [ ]:
# ============================================================
# LOAD WavLM ENCODER (FROZEN)
# ============================================================
from transformers import WavLMModel, Wav2Vec2FeatureExtractor

print('Loading WavLM (FROZEN)...')
wavlm = WavLMModel.from_pretrained('microsoft/wavlm-base-plus')
wavlm.eval()
for p in wavlm.parameters():
    p.requires_grad = False
wavlm = wavlm.to(device)
print(f'  WavLM: frozen, {sum(p.numel() for p in wavlm.parameters()):,} params')

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained('microsoft/wavlm-base-plus')
print('  Feature extractor ready')


In [ ]:
# ============================================================
# ATTENTION POOLING + MODEL
# ============================================================
import torch.nn as nn

class AttentionPooling(nn.Module):
    """Learns per-frame importance weights, then computes weighted sum."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
        
    def forward(self, hidden_states):
        # hidden_states: (batch, T, hidden_dim)
        weights = self.attn(hidden_states).squeeze(-1)  # (batch, T)
        weights = torch.softmax(weights, dim=-1)  # (batch, T)
        pooled = torch.bmm(weights.unsqueeze(1), hidden_states).squeeze(1)  # (batch, hidden_dim)
        return pooled, weights

class PhaseAWithProsody(nn.Module):
    """
    Phase A + Prosody: Simple and effective.
    
    Frozen WavLM → Attention Pooling → 768-dim audio representation
    + prosody_proj → 32-dim prosody representation
    → concat → MLP classifier
    """
    def __init__(self, encoder, prosody_dim=21, hidden=128):
        super().__init__()
        self.encoder = encoder
        self.audio_pool = AttentionPooling(768)
        
        # Prosody branch: 21-dim → 32-dim
        self.prosody_proj = nn.Sequential(
            nn.Linear(prosody_dim, 64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32)
        )
        
        # Total: 768 + 32 = 800
        total_dim = 768 + 32
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(total_dim, hidden),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, 64),
            nn.GELU(),
            nn.Linear(64, 2)
        )
        
    def forward(self, waveforms, prosody_feats):
        # waveforms: (batch, 1, 80000) — 5 seconds
        # prosody_feats: (batch, 21) — pre-extracted prosody
        
        # 1. WavLM encoder (frozen)
        wav_list = [w.cpu().numpy().squeeze() for w in waveforms]
        inputs = feature_extractor(wav_list, sampling_rate=SR, return_tensors='pt', padding=False)
        inputs = {k: v.to(waveforms.device) for k, v in inputs.items()}
        with torch.no_grad():
            hidden = self.encoder(**inputs).last_hidden_state  # (batch, T, 768)
        
        # 2. Attention pooling
        audio_emb, attn_weights = self.audio_pool(hidden)  # (batch, 768)
        
        # 3. Prosody projection
        prosody_emb = self.prosody_proj(prosody_feats)  # (batch, 32)
        
        # 4. Late fusion + classify
        fused = torch.cat([audio_emb, prosody_emb], dim=1)  # (batch, 800)
        logits = self.classifier(fused)
        
        return logits

model = PhaseAWithProsody(wavlm, prosody_dim=21, hidden=128).to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {n_params:,} total | {n_trainable:,} trainable (classifier only)')


In [ ]:
# ============================================================
# OPTIMIZER — Only classifier + prosody_proj trainable
# ============================================================
classifier_params = (
    list(model.audio_pool.parameters()) +
    list(model.prosody_proj.parameters()) +
    list(model.classifier.parameters())
)

optimizer = torch.optim.AdamW([
    {'params': classifier_params, 'lr': 1e-3, 'weight_decay': 0.01}
], lr=1e-3, weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

# Class weight: positive is ~29% in training data → weight = neg/pos
n_pos = sum(d['label_any'] for d in train)
n_neg = len(train) - n_pos
pos_weight = n_neg / n_pos
print(f'Class weight: neg={n_neg}, pos={n_pos}, ratio={pos_weight:.2f}')

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor([1.0, min(pos_weight, 4.0)]).to(device)
)
print(f'Loss: CrossEntropyLoss(weight=[1.0, {min(pos_weight, 4.0):.1f}])')


In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================
def run_epoch(model, data, optimizer, criterion, train=True, batch_size=BATCH, desc=''):
    if train:
        model.train()
        np.random.seed(SEED + (1 if train else 2))
        perm = np.random.permutation(len(data))
    else:
        model.eval()
        perm = np.arange(len(data))
    
    preds, labels_list = [], []
    total_loss = 0.0
    n_batches = (len(data) + batch_size - 1) // batch_size
    
    for bi in range(n_batches):
        idx = perm[bi*batch_size:(bi+1)*batch_size]
        batch = [data[i] for i in idx]
        
        # Prepare batch data
        audio_list, prosody_list, label_list = [], [], []
        for d in batch:
            key = f"{d['video_id']}_{d['start']:.2f}"
            audio = load_audio(key)
            pf = prosody_map.get(key, np.zeros(21))
            pf = prosody_scaler.transform(pf.reshape(1, -1)).flatten()
            audio_list.append(audio)
            prosody_list.append(pf)
            label_list.append(d['label_any'])
        
        audio_batch = np.stack(audio_list)
        prosody_batch = np.stack(prosody_list)
        label_batch = np.array(label_list)
        
        x = torch.from_numpy(audio_batch[:, None]).float().to(device)  # (B, 1, 80000)
        p = torch.from_numpy(prosody_batch).float().to(device)  # (B, 21)
        y = torch.from_numpy(label_batch).long().to(device)  # (B,)
        
        if train:
            optimizer.zero_grad()
        
        with torch.set_grad_enabled(train):
            logits = model(x, p)
            loss = criterion(logits, y)
        
        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(classifier_params, 1.0)
            optimizer.step()
        
        total_loss += loss.item()
        preds.extend(logits.argmax(1).cpu().tolist())
        labels_list.extend(y.cpu().tolist())
    
    avg_loss = total_loss / n_batches
    f1 = f1_score(labels_list, preds, zero_division=0)
    p = precision_score(labels_list, preds, zero_division=0)
    r = recall_score(labels_list, preds, zero_division=0)
    
    return avg_loss, f1, p, r, preds, labels_list

print(f'Training: {EPOCHS} epochs, batch={BATCH}')
print(f'Held-out test: Bill Burr, Dave Chappelle, Russell Peters')
print('='*70)

best_val_f1 = 0.0
best_state = None
history = []

for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    
    # Train
    train_loss, train_f1, train_p, train_r, _, _ = run_epoch(
        model, train, optimizer, criterion, train=True, desc=f'Epoch {epoch}')
    
    # Validate
    val_loss, val_f1, val_p, val_r, val_preds, val_labels = run_epoch(
        model, val, optimizer, criterion, train=False, desc='Val')
    
    scheduler.step()
    elapsed = time.time() - t0
    
    gap = train_f1 - val_f1
    star = ' ★' if val_f1 > best_val_f1 else ''
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
    print(f'Epoch {epoch:2d}/{EPOCHS} ({elapsed:.0f}s) | '
          f'Train F1={train_f1:.4f} P={train_p:.4f} R={train_r:.4f} | '
          f'Val F1={val_f1:.4f} P={val_p:.4f} R={val_r:.4f} | Gap={gap:.3f}{star}')
    
    history.append({
        'epoch': epoch, 'train_loss': train_loss, 'train_f1': train_f1,
        'train_p': train_p, 'train_r': train_r,
        'val_loss': val_loss, 'val_f1': val_f1,
        'val_p': val_p, 'val_r': val_r, 'time': elapsed
    })

print('='*70)
print(f'Best Val F1: {best_val_f1:.4f}')


In [ ]:
# ============================================================
# EVALUATE ON HELD-OUT TEST SET
# ============================================================
model.load_state_dict(best_state)
model.eval()

test_loss, test_f1, test_p, test_r, test_preds, test_labels = run_epoch(
    model, test, optimizer, criterion, train=False, batch_size=BATCH, desc='Test')

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(test_labels, test_preds)
tn, fp, fn, tp = cm.ravel()

print(f"\n{'='*60}")
print(f'HELD-OUT TEST (Bill Burr, Dave Chappelle, Russell Peters)')
print(f'  F1={test_f1:.4f} P={test_p:.4f} R={test_r:.4f}')
print(f'  TP={tp} FP={fp} FN={fn} TN={tn}')
print(f'  Positive ratio in test: {100*sum(test_labels)/len(test_labels):.1f}%')
print(f"{'='*60}")

# Save
torch.save({
    'epoch': epoch,
    'model_state_dict': best_state,
    'val_f1': best_val_f1,
    'test_f1': test_f1,
    'test_p': test_p,
    'test_r': test_r,
    'history': history,
    'config': {
        'architecture': 'PhaseA_Prosody_Fusion',
        'wavlm_frozen': True,
        'attention_pooling': True,
        'prosody_dim': 21,
        'class_weight': [1.0, min(pos_weight, 4.0)],
        'train_size': len(train),
        'val_size': len(val),
        'test_size': len(test),
    }
}, f'{OUTPUT_DIR}/phaseA_prosody_best.pt')
print(f'Saved: {OUTPUT_DIR}/phaseA_prosody_best.pt')

with open(f'{OUTPUT_DIR}/phaseA_prosody_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print(f'Saved: {OUTPUT_DIR}/phaseA_prosody_history.json')


In [ ]:
# ============================================================
# COMPARE: Phase A vs Phase A + Prosody
# ============================================================
# Phase A (frozen WavLM + attention pooling, no prosody): Val F1=0.756
# Phase A + Prosody: See above test_f1 result

print('Comparison:')
print(f'  Phase A (frozen WavLM + attn pool, NO prosody): Val F1=0.756 (in-distribution val)')
print(f'  Phase A + Prosody (this run):')
print(f'    Val F1={best_val_f1:.4f} (in-distribution val)')
print(f'    Test F1={test_f1:.4f} (held-out comedian test)')
print()
print('Key architectural differences from Phase D:')
print('  - No CSA (Compressed Sparse Attention) — it hurt ablation')
print('  - No Engram bottleneck — direct pooling is better')
print('  - WavLM FROZEN — partial unfreeze caused overfitting')
print('  - LR = 1e-3 only for head (was 5e-5 for encoder in Phase D)')
print('  - Class weight [1, 2.5] (was [1, 4])')
